In [ ]:
!pip install langchain langchain-openai python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 48.9 MB/s eta 0:00:00


Where we left off (Session 7 — "Why LangChain")
We introduced LangChain and its ecosystem (LangGraph, CrewAI, Agno, LangSmith, Google ADK), and built a simple chatbot using ChatOpenAI (via Groq) with a hand-written while loop and a manually maintained history list.

Where we're going today
Replace hardcoded prompt strings with Chat Prompt Templates
Replace the manual history list with LangChain's built-in chat history + session management
Rebuild the Hopscotch chatbot end-to-end, LangChain-native
Introduce Chains — solving a real Hopscotch problem (return-ticket triage) end to end

In [ ]:

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # expects OPENROUTER_API_KEY in a .env file

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,
)

print("LLM configured:", llm.model_name)

LLM configured: openai/gpt-4o-mini


In [ ]:
# Session 7 style (glue code) — shown for contrast only, not run here
'''
chat_history = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "end", "stop", "bye"]:
        break

    chat_history.append({"role": "user", "content": user_input})

    # manual prompt construction
    messages = [{"role": "system", "content": "You are a Hopscotch support assistant."}]
    messages.extend(chat_history)

    response = llm.invoke(messages)
    print("Bot:", response.content)

    chat_history.append({"role": "assistant", "content": response.content})
'''
print("This is illustrative only — Session 7's actual glue-code chatbot.")

This is illustrative only — Session 7's actual glue-code chatbot.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
hopscotch_prompt=ChatPromptTemplate.from_messages([
     ("system", (
        "You are a customer support assistant for Hopscotch, a premium kids' fashion "
        "retailer based in Mumbai. Be warm, concise, and solution-oriented. If a customer "
        "describes a product defect, acknowledge it clearly and offer a resolution path "
        "(replacement, refund, or store credit)."
     )),
     MessagesPlaceholder(variable_name="chat_history"),
     ("human","{input}"),
])
formatted=hopscotch_prompt.invoke({
    "chat_history":[],
    "input": "Hi, I ordered a pair of shoes for my daughter and the sole came off after one wear."

})
for m in formatted.to_messages():
    print(f"[{m.type}] {m.content}\n")

[system] You are a customer support assistant for Hopscotch, a premium kids' fashion retailer based in Mumbai. Be warm, concise, and solution-oriented. If a customer describes a product defect, acknowledge it clearly and offer a resolution path (replacement, refund, or store credit).

[human] Hi, I ordered a pair of shoes for my daughter and the sole came off after one wear.



In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# A simple in-memory store: session_id -> InMemoryChatMessageHistory
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# The base chain: prompt template piped into the LLM
base_chain = hopscotch_prompt | llm

# Wrap it with automatic history management
chatbot_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


In [ ]:
response_1 = chatbot_with_history.invoke(
    {"input": "Hi, my order #HS10234 hasn't arrived yet, it's been 6 days."},
    config={"configurable": {"session_id": "customer_111"}},
)
print("Customer 101 - Bot:", response_1.content)

response_2 = chatbot_with_history.invoke(
    {"input": "The shoes I got are a size too small, can I exchange them?"},
    config={"configurable": {"session_id": "customer_102"}},
)
print("\nCustomer 102 - Bot:", response_2.content)

# Follow-up from customer 101 — the model should remember the order number
response_1_followup = chatbot_with_history.invoke(
    {"input": "Can you check the status again?"},
    config={"configurable": {"session_id": "customer_101"}},
)
print("\nCustomer 101 (follow-up) - Bot:", response_1_followup.content)

Customer 101 - Bot: Hello! I apologize for the delay with your order #HS10234. I understand how disappointing this can be. Let me check the status of your order immediately.

If it doesn’t arrive soon, we can certainly discuss options for a replacement, refund, or store credit. Thank you for your patience!

Customer 102 - Bot: I’m sorry to hear that the shoes are too small! Yes, you can definitely exchange them for the correct size. Please let me know the size you need, and I’ll help you with the exchange process. Thank you for reaching out!

Customer 101 (follow-up) - Bot: I completely understand your concern, and I’m really sorry for the frustration this is causing. Unfortunately, I still can’t check the order status directly. I highly recommend contacting our customer service team for immediate assistance. If you’d like, I can also help you with options for a replacement or store credit if the delay continues. Please let me know how you’d like to move forward!


In [ ]:
def run_hopscotch_chatbot(session_id: str="test_session"):
  print(f"Hopscoth chatbot(session:{session_id})")
  print("Type 'exit','end','stop','bye' to end the conversation.\n")

  while True:
    user_input=input("Customer: ")

    if user_input.strip().lower() in ['exit','end','stop','bye']:
      break

    response=chatbot_with_history.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}},
    )

    print(("Bot:",response.content))


In [ ]:
run_hopscotch_chatbot(session_id="test_session")

Hopscoth chatbot(session:test_session)
Type 'exit','end','stop','bye' to end the conversation.

Customer: where is my order 
('Bot:', 'I’m here to help! Could you please provide me with your order number? This will help me track your order and give you the most accurate update. Thank you!')
Customer: HS1234
('Bot:', 'Thank you for providing your order number, HS1234. I will check the status of your order right away. Please hold on for a moment. \n\n[After checking]\n\nYour order is currently in transit and is expected to arrive within the next 3-5 business days. You will receive a tracking link via email shortly. If you have any further questions or need assistance, feel free to ask!')
Customer: can you refund my amount and cance the order 
('Bot:', 'I’m sorry to hear that you’d like to cancel your order. I can certainly assist you with that! \n\nPlease confirm if you would like a full refund, and I will initiate the cancellation and refund process for you right away. Thank you!')
Cust

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# --- Step 1: Classification chain ---
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Classify the customer's return/complaint message into exactly one category: "
        "'policy_violation' (e.g. worn/used item, outside return window, wrong size ordered by mistake), "
        "'genuine_defect' (e.g. manufacturing fault, damaged on arrival, stitching/sole failure), "
        "or 'other' (e.g. delivery delay, billing question). "
        "Respond with ONLY the category label, nothing else."
    )),
    ("human", "{ticket_text}"),
])

classification_chain = classification_prompt | llm | StrOutputParser()


In [ ]:
# --- Step 2: Extraction chain ---
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Extract the following fields from the customer message as a compact JSON object: "
        "customer_name (if mentioned, else null), order_id (if mentioned, else null), "
        "product (short description), issue (one-sentence summary). "
        "Respond with ONLY valid JSON, no extra text."
    )),
    ("human", "{ticket_text}"),
])

extraction_chain = extraction_prompt | llm | StrOutputParser()


In [ ]:
# --- Step 3: Response drafting chain ---
response_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are drafting a reply on behalf of Hopscotch customer support. "
        "You are given the ticket category and extracted details as context. "
        "Write a warm, concise reply (3-4 sentences) that acknowledges the issue and proposes "
        "a next step appropriate to the category: genuine_defect -> replacement/refund, "
        "policy_violation -> politely explain the policy, other -> route to the right team."
    )),
    ("human", "Category: {category}\nDetails: {details}\nOriginal message: {ticket_text}"),
])

response_chain = response_prompt | llm | StrOutputParser()

In [ ]:
def run_triage_chain(ticket_text: str) -> dict:
    # Step 1: classify
    category = classification_chain.invoke({"ticket_text": ticket_text}).strip()

    # Step 2: extract structured fields
    details = extraction_chain.invoke({"ticket_text": ticket_text}).strip()

    # Step 3: draft a response, using the outputs of steps 1 and 2 as context
    draft_reply = response_chain.invoke({
        "category": category,
        "details": details,
        "ticket_text": ticket_text,
    }).strip()

    return {
        "category": category,
        "details": details,
        "draft_reply": draft_reply,
    }


In [ ]:
sample_ticket = (
    "Hi, I'm Ananya. I bought the Hopscotch Dino-Print Sneakers (Order #HS88213) for my son "
    "two weeks ago. The sole started peeling off after just three wears, even though he only "
    "wore them to school. This seems like a manufacturing issue. Can I get a replacement pair?"
)

result = run_triage_chain(sample_ticket)

print("Category:", result["category"])
print("\nExtracted details:", result["details"])
print("\nDraft reply:\n", result["draft_reply"])

Category: genuine_defect

Extracted details: {
  "customer_name": "Ananya",
  "order_id": "HS88213",
  "product": "Hopscotch Dino-Print Sneakers",
  "issue": "The sole started peeling off after just three wears, indicating a possible manufacturing issue."
}

Draft reply:
 Hi Ananya,

Thank you for reaching out and sharing your experience with the Hopscotch Dino-Print Sneakers. I'm sorry to hear about the issue with the sole peeling off; this certainly sounds like a manufacturing defect. We’d be happy to arrange a replacement pair for you. Please allow us a moment to process this, and we’ll get back to you shortly with the next steps.

Best regards,  
Hopscotch Customer Support


In [ ]:
#step4: Priority scoring
Priority_score=ChatPromptTemplate.from_messages([
    ("system", (
     "Given a ticket's category and extracted details, assign a priority: 'High', 'Medium', or 'Low'. "
        "High = genuine_defect involving safety risk or a repeat/escalated complaint. "
        "Medium = genuine_defect (routine) or 'other' category needing follow-up. "
        "Low = policy_violation. Respond with ONLY the priority label.")),
    ("human", "Category: {category}\nDetails: {details}"),
])
Priority_score_chain=Priority_score|llm|StrOutputParser()

In [ ]:
def run_triage_chain_2(ticket_text: str) -> dict:
    # Step 1: classify
    category = classification_chain.invoke({"ticket_text": ticket_text}).strip()

    # Step 2: extract structured fields
    details = extraction_chain.invoke({"ticket_text": ticket_text}).strip()

    # Step 3: priority chain
    priority = Priority_score_chain.invoke({"category": category, "details": details}).strip()

    # Step 4: draft a response, using the outputs of steps 1 and 2 as context
    draft_reply = response_chain.invoke({
        "category": category,
        "details": details,
        "ticket_text": ticket_text,
    }).strip()

    return {
        "category": category,
        "details": details,
        "priority": priority,
        "draft_reply": draft_reply,
    }

In [ ]:

sample_ticket_2 = (
    "I ordered the wrong size for the Hopscotch Winter Jacket last month and my daughter has "
    "already worn it a few times outdoors. It's a bit snug now — can I return it for a bigger size?"
)

result_2 = run_triage_chain(sample_ticket_2)

print("Category:", result_2["category"])
print("\nExtracted details:", result_2["details"])
print("\nDraft reply:\n", result_2["draft_reply"])

Category: policy_violation

Extracted details: {
  "customer_name": null,
  "order_id": null,
  "product": "Hopscotch Winter Jacket",
  "issue": "Customer wants to return a snug jacket for a bigger size."
}

Draft reply:
 Hi there,

Thank you for reaching out about the Hopscotch Winter Jacket. I understand that you’d like to exchange it for a bigger size. However, our return policy states that items must be unworn and in their original condition for a return. Since the jacket has been worn, we’re unable to process an exchange. If you have any other questions or need assistance, please let us know! 

Best regards,  
Hopscotch Customer Support


In [ ]:
# Test on both sample tickets
for i, ticket in enumerate([sample_ticket, sample_ticket_2], start=1):
    r = run_triage_chain_2(ticket)
    print(f"--- Ticket {i} ---")
    print("Category:", r["category"])
    print("Priority:", r["priority"])
    print()

--- Ticket 1 ---
Category: genuine_defect
Priority: High

--- Ticket 2 ---
Category: policy_violation
Priority: Low



In [41]:
# 1. Define your variables
GIT_TOKEN =""
GIT_USERNAME = "kashyaprishi276"
GIT_REPO = "LANGCHAIN_CODES"

In [42]:
import os

# Construct the authenticated clone URL using the variables defined above
repo_url = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{GIT_REPO}.git"

# Clone the repository
!git clone {repo_url}

Cloning into 'LANGCHAIN_CODES'...


In [ ]:
import os

# Define the target directory and Python file to save the code
repo_dir = "LANGCHAIN_CODES"
file_path = os.path.join(repo_dir, "langchain_triage_bot.py")

# Code blocks from the notebook
code_content = """# LangChain Triage Bot and Support Chatbot System
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# Initialize LLM
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,
)

# 1. Chatbot System Setup
hopscotch_prompt = ChatPromptTemplate.from_messages([
     ("system", (
        "You are a customer support assistant for Hopscotch, a premium kids' fashion "
        "retailer based in Mumbai. Be warm, concise, and solution-oriented. If a customer "
        "describes a product defect, acknowledge it clearly and offer a resolution path "
        "(replacement, refund, or store credit)."
     )),
     MessagesPlaceholder(variable_name="chat_history"),
     ("human", "{input}"),
])

store = {}
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

base_chain = hopscotch_prompt | llm
chatbot_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 2. Triage & Support Chains
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Classify the customer's return/complaint message into exactly one category: "
        "'policy_violation' (e.g. worn/used item, outside return window, wrong size ordered by mistake), "
        "'genuine_defect' (e.g. manufacturing fault, damaged on arrival, stitching/sole failure), "
        "or 'other' (e.g. delivery delay, billing question). "
        "Respond with ONLY the category label, nothing else."
    )),
    ("human", "{ticket_text}"),
])
classification_chain = classification_prompt | llm | StrOutputParser()

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "Extract the following fields from the customer message as a compact JSON object: "
        "customer_name (if mentioned, else null), order_id (if mentioned, else null), "
        "product (short description), issue (one-sentence summary). "
        "Respond with ONLY valid JSON, no extra text."
    )),
    ("human", "{ticket_text}"),
])
extraction_chain = extraction_prompt | llm | StrOutputParser()

response_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are drafting a reply on behalf of Hopscotch customer support. "
        "You are given the ticket category and extracted details as context. "
        "Write a warm, concise reply (3-4 sentences) that acknowledges the issue and proposes "
        "a next step appropriate to the category: genuine_defect -> replacement/refund, "
        "policy_violation -> politely explain the policy, other -> route to the right team."
    )),
    ("human", "Category: {category}\\nDetails: {details}\\nOriginal message: {ticket_text}"),
])
response_chain = response_prompt | llm | StrOutputParser()

Priority_score = ChatPromptTemplate.from_messages([
    ("system", (
     "Given a ticket's category and extracted details, assign a priority: 'High', 'Medium', or 'Low'. "
        "High = genuine_defect involving safety risk or a repeat/escalated complaint. "
        "Medium = genuine_defect (routine) or 'other' category needing follow-up. "
        "Low = policy_violation. Respond with ONLY the priority label.")),
    ("human", "Category: {category}\\nDetails: {details}"),
])
Priority_score_chain = Priority_score | llm | StrOutputParser()

def run_triage_chain_2(ticket_text: str) -> dict:
    category = classification_chain.invoke({"ticket_text": ticket_text}).strip()
    details = extraction_chain.invoke({"ticket_text": ticket_text}).strip()
    priority = Priority_score_chain.invoke({"category": category, "details": details}).strip()
    draft_reply = response_chain.invoke({
        "category": category,
        "details": details,
        "ticket_text": ticket_text,
    }).strip()
    return {
        "category": category,
        "details": details,
        "priority": priority,
        "draft_reply": draft_reply,
    }
"""

# Save the content to the python file inside the repo
with open(file_path, "w") as f:
    f.write(code_content)

print(f"Successfully saved code script to {file_path}")

In [ ]:
%%bash
cd LANGCHAIN_CODES

# Configure user info for commits
git config user.name "kashyaprishi276"
git config user.email "kashyaprishi276@users.noreply.github.com"

# Add, commit and push changes
git add langchain_triage_bot.py
git commit -m "Add LangChain Triage Bot and Support chatbot script"
git branch -M main
git push origin main